In [6]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

print("=" * 60)
print("🗄️ CREATING SQL DATABASE")
print("=" * 60)

# 1) Ensure folders exist (remember: this notebook is in /notebooks/)
base_dir = os.path.abspath("..")  # one level up from /notebooks/
data_dir = os.path.join(base_dir, "data", "cleaned")
db_dir = os.path.join(base_dir, "database")

os.makedirs(db_dir, exist_ok=True)

print(f"\n📂 Project root: {base_dir}")
print(f"📂 Cleaned data folder: {data_dir}")
print(f"📂 Database folder: {db_dir}")

# 2) Load cleaned data
clean_file = os.path.join(data_dir, "data_analyst_jobs.csv")

if not os.path.exists(clean_file):
    raise FileNotFoundError(f"❌ Cleaned file not found: {clean_file}\n"
                            f"Make sure 02_data_cleaning.ipynb saved data_analyst_jobs.csv.")

print("\n1️⃣ Loading cleaned data...")
df = pd.read_csv(clean_file)
print(f"   ✅ Loaded {len(df)} rows")
print(f"   📊 Columns: {list(df.columns)}")

# 3) Create SQLite engine (database in project root /database/)
db_path = os.path.join(db_dir, "job_market.db")

# IMPORTANT: use forward slashes or escaped backslashes for SQLite URL
db_url = f"sqlite:///{db_path.replace('\\', '/')}"

print("\n2️⃣ Creating database engine...")
print(f"   🔗 DB URL: {db_url}")

engine = create_engine(db_url)

# 4) Save DataFrame to SQL
print("3️⃣ Saving data to SQL table 'jobs'...")
df.to_sql("jobs", engine, if_exists="replace", index=False)
print(f"   ✅ Saved {len(df)} rows into table 'jobs'")

# 5) Run some sample SQL queries to confirm

print("\n4️⃣ Running sample SQL queries...")

# Query 1: count rows
query1 = text("""
    SELECT COUNT(*) AS job_count
    FROM jobs
""")
with engine.connect() as conn:
    result1 = conn.execute(query1).fetchone()
    print(f"   📌 Total jobs in DB: {result1.job_count}")

# Query 2: salary by experience level
query2 = """
    SELECT 
        experience_level,
        COUNT(*) AS job_count,
        ROUND(AVG(salary_avg), 0) AS avg_salary
    FROM jobs
    GROUP BY experience_level
    ORDER BY avg_salary DESC
"""
result2 = pd.read_sql(query2, engine)
print("\n💰 Salary by experience level:")
print(result2.to_string(index=False))

# Query 3: jobs by city
query3 = """
    SELECT 
        location_clean AS city,
        COUNT(*) AS job_count,
        ROUND(AVG(salary_avg), 0) AS avg_salary
    FROM jobs
    GROUP BY location_clean
    ORDER BY job_count DESC
"""
result3 = pd.read_sql(query3, engine)
print("\n🏙️ Jobs by city:")
print(result3.head(10).to_string(index=False))

print("\n" + "=" * 60)
print("✅ SQL DATABASE CREATED AND VERIFIED!")
print("=" * 60)
print(f"\n📁 Database file location:\n   {db_path}")

🗄️ CREATING SQL DATABASE

📂 Project root: C:\Users\yelle\Job_Market_Analytics
📂 Cleaned data folder: C:\Users\yelle\Job_Market_Analytics\data\cleaned
📂 Database folder: C:\Users\yelle\Job_Market_Analytics\database

1️⃣ Loading cleaned data...
   ✅ Loaded 376 rows
   📊 Columns: ['job_id', 'job_title', 'company', 'location', 'salary_min', 'salary_max', 'job_type', 'experience_level', 'company_size', 'required_skills', 'posted_date', 'company_description', 'salary_avg', 'job_title_clean', 'location_clean', 'month', 'day_of_week', 'Python', 'SQL', 'Excel', 'Power BI', 'Tableau', 'Machine Learning', 'TensorFlow', 'PyTorch', 'AWS', 'Azure', 'Google Cloud', 'R', 'JavaScript', 'Docker', 'Kubernetes', 'Airflow', 'Spark', 'Hadoop', 'DAX', 'Visualization', 'Git', 'Jupyter', 'Pandas', 'NumPy', 'total_skills']

2️⃣ Creating database engine...
   🔗 DB URL: sqlite:///C:/Users/yelle/Job_Market_Analytics/database/job_market.db
3️⃣ Saving data to SQL table 'jobs'...
   ✅ Saved 376 rows into table 'jobs'